# Bias Detection & Fairness Analysis

This notebook implements the bias detection and fairness analysis for the NovaCred credit application dataset.
As part of the Data Governance Task Force, the goal is to identify potential discrimination in historical lending decisions
and quantify bias using established fairness metrics.

**Analyses performed:**
1. Gender Bias — Disparate Impact Ratio (four-fifths rule)
2. Proxy Discrimination — Geographic and demographic proxies for protected attributes
3. Age-Based Discrimination Patterns
4. Interaction Effects (gender × age, gender × region)
5. Fairlearn Fairness Metrics (demographic parity, equalized odds)

## 1. Data Loading and Preparation

We load the cleaned dataset produced by the Data Engineering pipeline (`01_data_quality.ipynb`).
Additional preparation includes standardizing gender labels and computing applicant age from date of birth.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set visualization style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# Load cleaned application data
df = pd.read_csv("../data/processed/credit_applications_clean.csv")
spend = pd.read_csv("../data/processed/spending_behavior_normalized.csv")

print(f"Dataset shape: {df.shape[0]} applications, {df.shape[1]} features")
df.head()

### 1.1 Gender Standardization

The Data Quality notebook identified inconsistent gender coding: both `Male`/`Female` and abbreviated `M`/`F` formats are present.
We standardize these for consistent analysis.

In [ ]:
# Inspect raw gender distribution before standardization
print("Raw gender values:")
print(df["applicant_info_gender"].value_counts(dropna=False))
print(f"\nMissing gender: {df['applicant_info_gender'].isna().sum()} records")

In [ ]:
# Standardize gender labels: M -> Male, F -> Female
df["gender"] = df["applicant_info_gender"].replace({"M": "Male", "F": "Female"})

print("Standardized gender distribution:")
print(df["gender"].value_counts(dropna=False))

# Drop rows with missing gender for bias analysis (2 records)
df_bias = df.dropna(subset=["gender"]).copy()
print(f"\nRecords for bias analysis: {len(df_bias)} (dropped {len(df) - len(df_bias)} with missing gender)")

### 1.2 Age Computation

We derive applicant age from the date of birth field, using January 2024 as the reference date
(consistent with the dataset's `processing_timestamp` values).

In [ ]:
# Parse date of birth and compute age
df_bias["date_of_birth"] = pd.to_datetime(df_bias["applicant_info_date_of_birth"], errors="coerce")
reference_date = pd.Timestamp("2024-01-01")
df_bias["age"] = ((reference_date - df_bias["date_of_birth"]).dt.days / 365.25).round(0)

print(f"Valid age values: {df_bias['age'].notna().sum()} / {len(df_bias)}")
print(f"Missing DOB: {df_bias['age'].isna().sum()} records\n")
print(df_bias["age"].describe())

### 1.3 Geographic Region Extraction

Zip codes in the dataset follow two distinct prefixes (`10xxx` and `90xxx`), suggesting two geographic regions.
We extract a region indicator for proxy discrimination analysis.

In [ ]:
# Extract region from zip code prefix
df_bias["zip_str"] = df_bias["applicant_info_zip_code"].astype(str).str.split(".").str[0]
df_bias["region"] = df_bias["zip_str"].str[:2].map({"10": "Region_10", "90": "Region_90"})

print("Region distribution:")
print(df_bias["region"].value_counts(dropna=False))

---
## 2. Gender Bias — Disparate Impact Ratio

The **Disparate Impact Ratio (DI)** measures whether a decision process disproportionately affects a protected group.
It is defined as:

$$DI = \frac{\text{Approval rate of unprivileged group}}{\text{Approval rate of privileged group}}$$

Under the **four-fifths rule**, a DI value below **0.8** indicates potential disparate impact and warrants further investigation.

In [ ]:
# Calculate approval rates by gender
approval_by_gender = df_bias.groupby("gender")["decision_loan_approved"].agg(["sum", "count", "mean"])
approval_by_gender.columns = ["approved", "total", "approval_rate"]
approval_by_gender["denied"] = approval_by_gender["total"] - approval_by_gender["approved"]

print("Approval rates by gender:")
print(approval_by_gender)
print()

In [ ]:
# Compute Disparate Impact Ratio
female_rate = approval_by_gender.loc["Female", "approval_rate"]
male_rate = approval_by_gender.loc["Male", "approval_rate"]

disparate_impact = female_rate / male_rate

print(f"Female approval rate: {female_rate:.4f} ({female_rate*100:.1f}%)")
print(f"Male approval rate:   {male_rate:.4f} ({male_rate*100:.1f}%)")
print(f"\nDisparate Impact Ratio: {disparate_impact:.4f}")
print(f"Four-fifths threshold:  0.8000")
print(f"\n{'⚠ POTENTIAL DISPARATE IMPACT DETECTED' if disparate_impact < 0.8 else '✓ No disparate impact detected'}")
print(f"The DI ratio of {disparate_impact:.4f} is {'below' if disparate_impact < 0.8 else 'above'} the 0.8 threshold.")

In [ ]:
# Statistical significance test: Chi-squared test for independence
contingency = pd.crosstab(df_bias["gender"], df_bias["decision_loan_approved"])
chi2, p_value, dof, expected = stats.chi2_contingency(contingency)

print("Chi-squared test of independence (gender × approval):")
print(f"  Chi² statistic: {chi2:.4f}")
print(f"  p-value:        {p_value:.6f}")
print(f"  Degrees of freedom: {dof}")
print(f"\n  Result: {'Statistically significant (p < 0.05)' if p_value < 0.05 else 'Not statistically significant'}")
print(f"  The association between gender and loan approval is {'significant' if p_value < 0.05 else 'not significant'}.")

In [ ]:
# Visualization: Approval rates by gender
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar chart of approval rates
colors = ["#e74c3c", "#3498db"]
approval_by_gender["approval_rate"].plot(kind="bar", ax=axes[0], color=colors, edgecolor="black")
axes[0].axhline(y=male_rate * 0.8, color="red", linestyle="--", linewidth=1.5, label=f"4/5 threshold ({male_rate*0.8:.2f})")
axes[0].set_title("Loan Approval Rate by Gender", fontsize=14, fontweight="bold")
axes[0].set_ylabel("Approval Rate")
axes[0].set_xlabel("Gender")
axes[0].set_ylim(0, 1)
axes[0].legend()
axes[0].tick_params(axis="x", rotation=0)

# Annotate bars
for i, (gender, row) in enumerate(approval_by_gender.iterrows()):
    axes[0].text(i, row["approval_rate"] + 0.02, f"{row['approval_rate']:.1%}", ha="center", fontweight="bold")

# Stacked bar chart of approved vs denied
approval_by_gender[["approved", "denied"]].plot(kind="bar", stacked=True, ax=axes[1],
                                                 color=["#2ecc71", "#e74c3c"], edgecolor="black")
axes[1].set_title("Approved vs Denied by Gender", fontsize=14, fontweight="bold")
axes[1].set_ylabel("Number of Applications")
axes[1].set_xlabel("Gender")
axes[1].legend(["Approved", "Denied"])
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.savefig("../reports/gender_approval_rates.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nDisparate Impact Ratio = {disparate_impact:.4f} (threshold: 0.8)")